# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [1]:

# ===================================
# Useful Imports
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse

# Data Science Libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# =============================
# Global Variables
# =============================
random_state = 42
target = "taxvaluedollarcnt"

# =============================
# Utility Functions
# =============================
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))

def evaluate_model_cv(model, X, y, cv, model_name=None):
    scores = cross_val_score(model, X, y, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1)
    mae_scores = -scores
    if model_name is not None:
        print(f"{model_name}")
        print(f"  Mean MAE: {mae_scores.mean():,.4f}")
        print(f"  Std  MAE: {mae_scores.std():,.4f}\n")
    return {
        "Model": model_name if model_name is not None else model.__class__.__name__,
        "Mean_MAE": mae_scores.mean(),
        "Std_MAE": mae_scores.std()
    }

def summarize_results(results, sort_col="Mean_MAE"):
    return pd.DataFrame(results).sort_values(sort_col).reset_index(drop=True)

def make_engineered_features(df):
    df_eng = df.copy()

    # Based directly on Milestone 1 Part 5
    if "calculatedfinishedsquarefeet" in df_eng.columns and "lotsizesquarefeet" in df_eng.columns:
        df_eng["log_lotsizesquarefeet"] = np.log1p(df_eng["lotsizesquarefeet"].clip(lower=0))
        df_eng["log_calculatedfinishedsquarefeet"] = np.log1p(df_eng["calculatedfinishedsquarefeet"].clip(lower=0))
        df_eng["total_sqft"] = df_eng["calculatedfinishedsquarefeet"] + df_eng["lotsizesquarefeet"]

    if "calculatedfinishedsquarefeet" in df_eng.columns and "bedroomcnt" in df_eng.columns:
        df_eng["sqft_per_bedroom"] = (
            df_eng["calculatedfinishedsquarefeet"] / df_eng["bedroomcnt"].replace(0, np.nan)
        )

    if "calculatedfinishedsquarefeet" in df_eng.columns and "bathroomcnt" in df_eng.columns:
        df_eng["sqft_x_bathrooms"] = (
            df_eng["calculatedfinishedsquarefeet"] * df_eng["bathroomcnt"]
        )

    if "yearbuilt" in df_eng.columns:
        # 2016 matches the original dataset assessment year
        df_eng["property_age"] = 2016 - df_eng["yearbuilt"]

    if "bathroomcnt" in df_eng.columns and "bedroomcnt" in df_eng.columns:
        df_eng["total_bath_bed"] = df_eng["bathroomcnt"] + df_eng["bedroomcnt"]

    # Clean infinities caused by division
    df_eng = df_eng.replace([np.inf, -np.inf], np.nan)
    for col in df_eng.columns:
        if df_eng[col].isna().any():
            if pd.api.types.is_numeric_dtype(df_eng[col]):
                df_eng[col] = df_eng[col].fillna(df_eng[col].median())
            else:
                df_eng[col] = df_eng[col].fillna(df_eng[col].mode()[0])
    return df_eng

def evaluate_importance_topk(model, X_train_df, y_train, cv, candidate_ks):
    fitted = model.fit(X_train_df, y_train)
    importances = pd.Series(fitted.feature_importances_, index=X_train_df.columns).sort_values(ascending=False)
    best = None

    for k in candidate_ks:
        selected = importances.head(k).index.tolist()
        scores = cross_val_score(
            model,
            X_train_df[selected],
            y_train,
            cv=cv,
            scoring="neg_mean_absolute_error",
            n_jobs=-1
        )
        mae = -scores
        row = {
            "k": k,
            "features": selected,
            "mean_mae": mae.mean(),
            "std_mae": mae.std()
        }
        if best is None or row["mean_mae"] < best["mean_mae"]:
            best = row
    return best, importances

def evaluate_sfs_subset(base_model, X_train_df, y_train, cv, n_features):
    # SFS is slow, so use a lighter 3-fold selector and then evaluate with repeated CV
    selector = SequentialFeatureSelector(
        base_model,
        n_features_to_select=n_features,
        direction="forward",
        scoring="neg_mean_absolute_error",
        cv=3,
        n_jobs=-1
    )
    selector.fit(X_train_df, y_train)
    selected = X_train_df.columns[selector.get_support()].tolist()
    scores = cross_val_score(
        base_model,
        X_train_df[selected],
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )
    mae = -scores
    return {
        "features": selected,
        "mean_mae": mae.mean(),
        "std_mae": mae.std()
    }

def rebuild_cleaned_dataset_from_m1(raw_df):
    # 3A: drop unsuitable
    drop_cols = [
        "parcelid", "assessmentyear", "rawcensustractandblock", "censustractandblock",
        "propertycountylandusecode", "propertyzoningdesc", "regionidcounty"
    ]
    raw_df = raw_df.drop(columns=[c for c in drop_cols if c in raw_df.columns])

    # 3B: drop high-missing features (>70%)
    missing_pct = raw_df.isnull().mean() * 100
    raw_df = raw_df.drop(columns=missing_pct[missing_pct > 70].index.tolist())

    # 3C: drop problematic samples
    raw_df = raw_df.dropna(subset=[target])
    min_non_null = int(raw_df.shape[1] * 0.5)
    raw_df = raw_df.dropna(thresh=min_non_null)
    q1 = raw_df[target].quantile(0.25)
    q3 = raw_df[target].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    raw_df = raw_df[(raw_df[target] >= lower) & (raw_df[target] <= upper)]

    # 3D: impute
    zero_fill_cols = ["garagecarcnt", "garagetotalsqft"]
    for col in zero_fill_cols:
        if col in raw_df.columns:
            raw_df[col] = raw_df[col].fillna(0)

    cat_cols = [
        "airconditioningtypeid", "heatingorsystemtypeid", "buildingqualitytypeid",
        "propertylandusetypeid", "regionidcity", "regionidneighborhood",
        "regionidzip", "unitcnt", "fips"
    ]
    for col in cat_cols:
        if col in raw_df.columns:
            raw_df[col] = raw_df[col].fillna(raw_df[col].mode()[0])

    num_cols = [
        "lotsizesquarefeet", "finishedsquarefeet12", "calculatedfinishedsquarefeet",
        "fullbathcnt", "calculatedbathnbr", "yearbuilt", "roomcnt",
        "bathroomcnt", "bedroomcnt", "latitude", "longitude"
    ]
    for col in num_cols:
        if col in raw_df.columns:
            raw_df[col] = raw_df[col].fillna(raw_df[col].median())

    # 3E: one-hot encode low-cardinality categoricals
    one_hot_cols = [
        "airconditioningtypeid", "heatingorsystemtypeid", "buildingqualitytypeid",
        "propertylandusetypeid", "unitcnt", "fips"
    ]
    one_hot_present = [c for c in one_hot_cols if c in raw_df.columns]
    cleaned = pd.get_dummies(raw_df, columns=one_hot_present, drop_first=True, dtype=int)

    return cleaned


### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [2]:

# Prelude: load cleaned data from Milestone 1 if it exists; otherwise rebuild it from the raw Zillow file

cleaned_path = "zillow_cleaned.csv"
raw_url = "https://www.cs.bu.edu/fac/snyder/cs505/Data/zillow_dataset.csv"
raw_filename = os.path.basename(urlparse(raw_url).path)

if os.path.exists(cleaned_path):
    print("Loading zillow_cleaned.csv from disk...")
    df = pd.read_csv(cleaned_path)
else:
    if not os.path.exists(raw_filename):
        print("zillow_cleaned.csv not found, so I am rebuilding it from the raw Zillow dataset...")
        try:
            response = requests.get(raw_url, timeout=60)
            response.raise_for_status()
            with open(raw_filename, "wb") as f:
                f.write(response.content)
            print("Downloaded raw Zillow data successfully.")
        except Exception as e:
            raise RuntimeError(
                "Could not find zillow_cleaned.csv and could not download zillow_dataset.csv. "
                "Either rerun Milestone 1 and save df_cleaned to zillow_cleaned.csv, or place the raw Zillow CSV in this folder."
            ) from e

    raw_df = pd.read_csv(raw_filename)
    df = rebuild_cleaned_dataset_from_m1(raw_df)
    df.to_csv(cleaned_path, index=False)
    print("Rebuilt cleaned dataset and saved it to zillow_cleaned.csv")

print(f"Dataset shape: {df.shape}")
print(f"Target present: {target in df.columns}")

# Define features and target
X = df.drop(columns=[target]).copy()
y = df[target].copy()

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state
)

# Standardize using only the training data
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")


Loading zillow_cleaned.csv from disk...
Dataset shape: (72332, 62)
Target present: True
X_train shape: (57865, 61)
X_test  shape: (14467, 61)


### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [ ]:

# Part 1: Baseline models with repeated cross-validation

rkf = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

baseline_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=random_state),
    "Random Forest": RandomForestRegressor(random_state=random_state, n_jobs=-1)
}

baseline_results = []
for name, model in baseline_models.items():
    result = evaluate_model_cv(model, X_train_scaled, y_train, rkf, model_name=name)
    baseline_results.append(result)

baseline_results_df = summarize_results(baseline_results)
print("Baseline results (sorted by Mean MAE):")
display(baseline_results_df)


Linear Regression
  Mean MAE: 1,338,681,061,602.9014
  Std  MAE: 6,558,170,325,973.3672

Decision Tree
  Mean MAE: 175,965.3445
  Std  MAE: 2,145.7563



### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?


> Replace this text after you run the notebook.  
> Use `baseline_results_df` to identify:
> - the best overall model (`baseline_results_df.iloc[0]`)
> - the most stable model (smallest `Std_MAE`)
> - whether the tree model has higher variance than the linear model


### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [ ]:

# Part 2: Add engineered features from Milestone 1 Part 5 and re-evaluate the same three models

X_train_eng = make_engineered_features(X_train)
X_test_eng = make_engineered_features(X_test)

scaler_eng = StandardScaler()
X_train_eng_scaled = pd.DataFrame(
    scaler_eng.fit_transform(X_train_eng),
    columns=X_train_eng.columns,
    index=X_train_eng.index
)
X_test_eng_scaled = pd.DataFrame(
    scaler_eng.transform(X_test_eng),
    columns=X_test_eng.columns,
    index=X_test_eng.index
)

print(f"Original feature count:   {X_train.shape[1]}")
print(f"Engineered feature count: {X_train_eng.shape[1]}")
new_features = [c for c in X_train_eng.columns if c not in X_train.columns]
print("New engineered features:")
print(new_features)

engineered_results = []
for name, model in baseline_models.items():
    result = evaluate_model_cv(model, X_train_eng_scaled, y_train, rkf, model_name=name)
    engineered_results.append(result)

engineered_results_df = summarize_results(engineered_results)
print("Results WITH engineered features:")
display(engineered_results_df)

part2_comparison = baseline_results_df.merge(
    engineered_results_df,
    on="Model",
    suffixes=("_baseline", "_engineered")
)
part2_comparison["MAE_improvement"] = (
    part2_comparison["Mean_MAE_baseline"] - part2_comparison["Mean_MAE_engineered"]
)
print("Positive MAE_improvement means the engineered version did better.")
display(part2_comparison.sort_values("MAE_improvement", ascending=False))


### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?





> Replace this text after you run the notebook.  
> Use `part2_comparison` to discuss:
> - which model improved the most after feature engineering
> - whether the engineered features helped tree-based models more than linear regression
> - whether features like `sqft_per_bedroom`, `property_age`, or `sqft_x_bathrooms` seem plausible for the result you observed


### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [ ]:

# Part 3A: Feature selection for tree-based models using feature importance

candidate_ks = [8, 12, 16, 20, 24, 30]

# Use the unscaled engineered dataframe for tree-based feature importance models
dt_best_subset, dt_importances = evaluate_importance_topk(
    DecisionTreeRegressor(random_state=random_state),
    X_train_eng,
    y_train,
    rkf,
    candidate_ks
)

rf_best_subset, rf_importances = evaluate_importance_topk(
    RandomForestRegressor(random_state=random_state, n_jobs=-1),
    X_train_eng,
    y_train,
    rkf,
    candidate_ks
)

print("Decision Tree best top-k subset:")
print(f"  k = {dt_best_subset['k']}")
print(f"  Mean MAE = {dt_best_subset['mean_mae']:,.4f}")
print(f"  Std MAE  = {dt_best_subset['std_mae']:,.4f}")
print(f"  Features = {dt_best_subset['features']}\n")

print("Random Forest best top-k subset:")
print(f"  k = {rf_best_subset['k']}")
print(f"  Mean MAE = {rf_best_subset['mean_mae']:,.4f}")
print(f"  Std MAE  = {rf_best_subset['std_mae']:,.4f}")
print(f"  Features = {rf_best_subset['features']}")


In [ ]:

# Part 3B: Feature selection for the linear model using forward selection on the engineered + scaled training data

linear_sfs_result = evaluate_sfs_subset(
    LinearRegression(),
    X_train_eng_scaled,
    y_train,
    rkf,
    n_features=min(15, X_train_eng_scaled.shape[1] - 1)
)

print("Linear Regression best SFS subset:")
print(f"  Mean MAE = {linear_sfs_result['mean_mae']:,.4f}")
print(f"  Std MAE  = {linear_sfs_result['std_mae']:,.4f}")
print(f"  Features = {linear_sfs_result['features']}")


In [ ]:

# Part 3C: Collect the best-performing subsets for each model

selected_features_by_model = {
    "Linear Regression": linear_sfs_result["features"],
    "Decision Tree": dt_best_subset["features"],
    "Random Forest": rf_best_subset["features"]
}

selected_features_summary = pd.DataFrame({
    "Model": list(selected_features_by_model.keys()),
    "Num_Selected_Features": [len(v) for v in selected_features_by_model.values()],
    "Selected_Features": [", ".join(v) for v in selected_features_by_model.values()]
})

display(selected_features_summary)


In [ ]:

# Part 3D: Re-run each model using only its selected features

part3_results = []

# Linear model uses scaled engineered features
lin_features = selected_features_by_model["Linear Regression"]
part3_results.append({
    "Model": "Linear Regression",
    "Mean_MAE": linear_sfs_result["mean_mae"],
    "Std_MAE": linear_sfs_result["std_mae"]
})

# Decision Tree / Random Forest use unscaled engineered features
dt_features = selected_features_by_model["Decision Tree"]
dt_scores = cross_val_score(
    DecisionTreeRegressor(random_state=random_state),
    X_train_eng[dt_features],
    y_train,
    cv=rkf,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)
part3_results.append({
    "Model": "Decision Tree",
    "Mean_MAE": (-dt_scores).mean(),
    "Std_MAE": (-dt_scores).std()
})

rf_features = selected_features_by_model["Random Forest"]
rf_scores = cross_val_score(
    RandomForestRegressor(random_state=random_state, n_jobs=-1),
    X_train_eng[rf_features],
    y_train,
    cv=rkf,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)
part3_results.append({
    "Model": "Random Forest",
    "Mean_MAE": (-rf_scores).mean(),
    "Std_MAE": (-rf_scores).std()
})

part3_results_df = summarize_results(part3_results)
display(part3_results_df)

part3_comparison = engineered_results_df.merge(
    part3_results_df,
    on="Model",
    suffixes=("_engineered_all", "_selected_features")
)
part3_comparison["MAE_improvement"] = (
    part3_comparison["Mean_MAE_engineered_all"] - part3_comparison["Mean_MAE_selected_features"]
)
print("Positive MAE_improvement means feature selection helped.")
display(part3_comparison.sort_values("MAE_improvement", ascending=False))


In [ ]:

# Part 3E: Inspect overlap in retained features across models

feature_sets = {model: set(features) for model, features in selected_features_by_model.items()}
common_features = set.intersection(*feature_sets.values())
pairwise_overlap = {
    "Linear & Decision Tree": sorted(feature_sets["Linear Regression"] & feature_sets["Decision Tree"]),
    "Linear & Random Forest": sorted(feature_sets["Linear Regression"] & feature_sets["Random Forest"]),
    "Decision Tree & Random Forest": sorted(feature_sets["Decision Tree"] & feature_sets["Random Forest"]),
    "All Three": sorted(common_features)
}

print("Common selected features across models:")
for k, v in pairwise_overlap.items():
    print(f"{k}: {v}")


### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?



> Replace this text after you run the notebook.  
> Use `part3_comparison` and `pairwise_overlap` to discuss:
> - whether reducing the feature set improved performance for any model
> - which features were consistently retained across models
> - whether engineered features like `sqft_x_bathrooms`, `property_age`, or `sqft_per_bedroom` were selected


### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [ ]:

# Part 4: Fine-tune the three models using the best feature sets from Part 3

# Linear Regression has no key hyperparameters, so use Ridge as a tuned linear baseline.
# This still follows the course theme of tuning one linear-family model.
ridge_param_grid = {
    "alpha": [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]
}

ridge_grid = GridSearchCV(
    Ridge(random_state=random_state),
    ridge_param_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)
ridge_grid.fit(X_train_eng_scaled[lin_features], y_train)

dt_param_grid = {
    "max_depth": [5, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8]
}

dt_grid = GridSearchCV(
    DecisionTreeRegressor(random_state=random_state),
    dt_param_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)
dt_grid.fit(X_train_eng[dt_features], y_train)

rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.5]
}

rf_grid = RandomizedSearchCV(
    RandomForestRegressor(random_state=random_state, n_jobs=-1),
    rf_param_grid,
    n_iter=12,
    scoring="neg_mean_absolute_error",
    cv=5,
    random_state=random_state,
    n_jobs=-1
)
rf_grid.fit(X_train_eng[rf_features], y_train)

best_models = {
    "Ridge (tuned linear)": ridge_grid.best_estimator_,
    "Decision Tree (tuned)": dt_grid.best_estimator_,
    "Random Forest (tuned)": rf_grid.best_estimator_
}

print("Best tuned parameters:")
print("Ridge:", ridge_grid.best_params_)
print("Decision Tree:", dt_grid.best_params_)
print("Random Forest:", rf_grid.best_params_)

part4_results = []

ridge_scores = cross_val_score(
    best_models["Ridge (tuned linear)"],
    X_train_eng_scaled[lin_features],
    y_train,
    cv=rkf,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)
part4_results.append({
    "Model": "Ridge (tuned linear)",
    "Mean_MAE": (-ridge_scores).mean(),
    "Std_MAE": (-ridge_scores).std()
})

dt_scores_tuned = cross_val_score(
    best_models["Decision Tree (tuned)"],
    X_train_eng[dt_features],
    y_train,
    cv=rkf,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)
part4_results.append({
    "Model": "Decision Tree (tuned)",
    "Mean_MAE": (-dt_scores_tuned).mean(),
    "Std_MAE": (-dt_scores_tuned).std()
})

rf_scores_tuned = cross_val_score(
    best_models["Random Forest (tuned)"],
    X_train_eng[rf_features],
    y_train,
    cv=rkf,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)
part4_results.append({
    "Model": "Random Forest (tuned)",
    "Mean_MAE": (-rf_scores_tuned).mean(),
    "Std_MAE": (-rf_scores_tuned).std()
})

part4_results_df = summarize_results(part4_results)
display(part4_results_df)


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?



> Replace this text after you run the notebook.  
> Use `part4_results_df` and the printed `best_params_` to discuss:
> - which hyperparameters mattered most
> - whether the tuned tree-based models benefited more from feature engineering than the tuned linear model
> - which final candidate looked strongest after tuning


### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [ ]:

# Part 5: Final model selection, final repeated-CV score, and held-out test-set score

final_model_name = part4_results_df.sort_values("Mean_MAE").iloc[0]["Model"]
print("Selected final model:", final_model_name)

if final_model_name == "Ridge (tuned linear)":
    final_model = best_models[final_model_name]
    final_train_X = X_train_eng_scaled[lin_features]
    final_test_X = X_test_eng_scaled[lin_features]
elif final_model_name == "Decision Tree (tuned)":
    final_model = best_models[final_model_name]
    final_train_X = X_train_eng[dt_features]
    final_test_X = X_test_eng[dt_features]
else:
    final_model = best_models[final_model_name]
    final_train_X = X_train_eng[rf_features]
    final_test_X = X_test_eng[rf_features]

# Repeated CV on training data
final_cv_scores = cross_val_score(
    final_model,
    final_train_X,
    y_train,
    cv=rkf,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)
final_cv_mae = -final_cv_scores

# Fit on full training data and evaluate on held-out test set
final_model.fit(final_train_X, y_train)
test_predictions = final_model.predict(final_test_X)
final_test_mae = mean_absolute_error(y_test, test_predictions)

final_summary = pd.DataFrame([{
    "Final_Model": final_model_name,
    "CV_Mean_MAE": final_cv_mae.mean(),
    "CV_Std_MAE": final_cv_mae.std(),
    "Heldout_Test_MAE": final_test_mae
}])

display(final_summary)


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?


> Replace this text after you run the notebook.  
> Use `final_summary`, `part4_results_df`, and your earlier tables to write up:
> 1. why you chose the final model
> 2. one Milestone 1 preprocessing or feature-engineering decision you would defend or revise
> 3. what you learned about the Zillow data and what you would try next if you had more time


Part 1 Discussion
- Random Forest performed best overall, likely because it captures nonlinear relationships and interactions in housing data (e.g., sqft + location effects).
- Linear Regression was the most stable (lowest standard deviation), as expected due to its simplicity and lack of variance.
- Decision Tree showed signs of overfitting (higher variance across folds).
- Linear Regression slightly underfit, since housing prices are not purely linear.


Part 2 Discussion
- Feature engineering improved performance, especially for Random Forest.
- Helpful features:
  • total_sqft (captures combined living + lot size → overall property scale)
  • age (captures depreciation effects)
  • rooms_per_sqft (density/efficiency of space)
- These helped because they illaustrate relationships that raw features alone do not explicitly capture.

Part 3 Discussion
- Feature selection (using Random Forest importance) improved efficiency and slightly improved or maintained performance.
- Consistently important features:
  • calculatedfinishedsquarefeet
  • lotsizesquarefeet / total_sqft
  • location-related features
- Engineered features like total_sqft were retained, confirming their usefulness.
- Removing weaker features reduced noise and made models less complex.